# Proyecto 2 Entrega de Ingeniería de datos

Trabajamos con los anuncios de Airbnb de San Diego. Preparamos una base para comparar alojamientos y otra con precios publicados por noche para el análisis de regresión. Cada fila corresponde a un anuncio.

## 0. Librerías y carga de datos

Usamos el archivo detallado de Inside Airbnb que se descargó.

In [2]:
from pathlib import Path
import hashlib
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

carpeta_actual = Path.cwd()
raiz = carpeta_actual if (carpeta_actual / "listings.csv").is_file() else carpeta_actual.parent
entrada = raiz / "listings.csv"
if not entrada.is_file():
    raise FileNotFoundError("Abre el notebook desde el proyecto o su carpeta ingenieria_datos.")

salida = raiz / "ingenieria_datos" / "datos_procesados"
ids_texto = {c: "string" for c in ["id", "host_id", "host_profile_id", "scrape_id"]}
marcadores = ["", "n", "na", "n/a", "nan", "none", "null"]
original = pd.read_csv(entrada, dtype=ids_texto, low_memory=False,
                       na_values=["N", "n", "N/A", "NULL", "null", "None"])
print(f"{entrada.name}: {original.shape[0]:,} anuncios y {original.shape[1]} columnas.")
print("Capturas:", original["last_scraped"].min(), "a", original["last_scraped"].max())
display(original[["neighbourhood_cleansed", "room_type", "accommodates", "price"]].head())

listings.csv: 13,253 anuncios y 90 columnas.
Capturas: 2026-09-21 a 2026-09-23


,neighbourhood_cleansed,room_type,accommodates,price
0,North Hills,Entire home/apt,7,$316.00
1,Pacific Beach,Private room,2,$146.50
2,Pacific Beach,Private room,2,$121.00
3,North Hills,Private room,2,NaN
4,Pacific Beach,Entire home/apt,2,$282.00


## 1. Exploración inicial

Revisamos los tipos de datos y contamos los valores que faltan en cada columna. También miramos cuántos anuncios tienen precio por tipo de habitación.

In [3]:
perfil = pd.DataFrame({
    "tipo_leído": original.dtypes.astype(str),
    "nulos": original.isna().sum(),
    "nulos_%": original.isna().mean().mul(100).round(2),
    "valores_distintos": original.nunique()
}).sort_values("nulos", ascending=False, kind="stable")
print(f"Columnas completamente vacías: {original.isna().all().sum()}")
with pd.option_context("display.max_rows", len(perfil)):
    display(perfil)

display(original[["latitude", "longitude", "accommodates", "bedrooms", "beds",
                  "minimum_nights", "availability_365", "number_of_reviews",
                  "review_scores_rating"]].describe().T.round(2))

Columnas completamente vacías: 14


,tipo_leído,nulos,nulos_%,valores_distintos
neighborhood_overview,float64,13253,100.00,0
host_since,float64,13253,100.00,0
host_response_time,float64,13253,100.00,0
host_response_rate,float64,13253,100.00,0
host_acceptance_rate,float64,13253,100.00,0
host_thumbnail_url,float64,13253,100.00,0
host_neighbourhood,float64,13253,100.00,0
host_total_listings_count,float64,13253,100.00,0
host_verifications,float64,13253,100.00,0
neighbourhood,float64,13253,100.00,0


,count,mean,std,min,25%,50%,75%,max
latitude,13253.0,32.77,0.07,32.54,32.73,32.76,32.80,33.09
longitude,13253.0,-117.18,0.07,-117.28,-117.25,-117.17,-117.14,-116.94
accommodates,13253.0,5.15,3.37,1.00,2.00,4.00,6.00,16.00
bedrooms,13229.0,1.96,1.42,0.00,1.00,2.00,3.00,32.00
beds,12880.0,2.89,2.37,1.00,1.00,2.00,4.00,37.00
minimum_nights,13171.0,11.80,21.39,1.00,2.00,2.00,31.00,730.00
availability_365,13253.0,217.94,121.64,0.00,106.00,252.00,329.00,365.00
number_of_reviews,13253.0,73.78,120.45,0.00,3.00,25.00,93.00,1918.00
review_scores_rating,11099.0,4.81,0.33,1.00,4.77,4.90,4.98,5.00


In [4]:
cobertura = original.groupby("room_type")["price"].agg(anuncios="size", con_precio="count")
cobertura["sin_precio"] = cobertura["anuncios"] - cobertura["con_precio"]
cobertura["cobertura_%"] = (100 * cobertura["con_precio"] / cobertura["anuncios"]).round(1)
display(cobertura)
display(original.groupby(original["price"].notna().map({True: "Con precio", False: "Sin precio"}))
        [["availability_365", "number_of_reviews"]].median()
        .rename_axis("Precio publicado").rename(columns={"availability_365": "Mediana días disponibles",
                                                         "number_of_reviews": "Mediana reseñas"}))

,anuncios,con_precio,sin_precio,cobertura_%
room_type,,,,
Entire home/apt,11052,9764,1288,88.3
Hotel room,277,267,10,96.4
Private room,1872,1541,331,82.3
Shared room,52,42,10,80.8


,Mediana días disponibles,Mediana reseñas
Precio publicado,,
Con precio,264.0,30.0
Sin precio,0.0,6.0


## 2. Selección de variables

Para estudiar el precio usamos el barrio, el tipo de alojamiento, la capacidad y los servicios. Guardamos las reseñas y los días disponibles para comparar los anuncios.

Quitamos nombres, enlaces y columnas vacías. También dejamos por fuera los ingresos estimados porque pueden revelar el precio.

In [5]:
columnas = [
    "id", "host_id", "last_scraped", "neighbourhood_cleansed", "room_type", "property_type",
    "latitude", "longitude", "accommodates", "bedrooms", "beds", "bathrooms_text", "amenities",
    "host_is_superhost", "minimum_nights", "availability_365", "number_of_reviews",
    "last_review", "review_scores_rating", "price"
]
d = original[columnas].copy()
# Quitamos las filas que se repiten por completo.
d = d.loc[~original.duplicated()].copy()
if d["id"].isna().any() or not d["id"].is_unique:
    raise ValueError("Revisar registros sin ID o con el mismo ID antes de consolidar el corte.")
if d["host_id"].isna().any():
    raise ValueError("Falta el anfitrión necesario para la partición por grupos.")